# 머신러닝과 성능 평가

독창적 가상 데이터와 자체 연습 문항입니다. 제공 샘플의 흐름을 참고했으며 원문 복제가 아닙니다.

14문항 · 파트 연습 · 힌트 사용 가능

CSV 두 개와 노트북을 같은 폴더에 두세요. 커널 재시작 → 전체 실행 → 저장한 뒤 사이트에 제출하세요. 각 답안 다음 검증 셀은 실행 결과를 검사합니다. 검증 셀을 바꾸면 점수 신뢰도가 사라집니다. 공식 점수 예측이 아닌 자체 연습 점수입니다.

데이터: ticket_id 식별번호, created_at 접수일시, issue_type 문의유형, contact_channel 채널, membership_tier 등급, tenure_months 가입개월, monthly_spend 월금액(원), previous_tickets 과거 문의 수, wait_minutes 대기분, past_satisfaction_score 과거만족도(1~5), escalated 이관여부(0/1), resolution_hours 해결시간(시간). resolution_hours는 분류 입력에서 제외합니다.

In [106]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, confusion_matrix, mean_absolute_error
SET_ID='part-ml'
DATA_FILE="practice.csv"
MEMBERS_FILE="members.csv"
MODEL_FILE="best_part-ml.keras"

from pathlib import Path
import json as _json
from IPython.display import display as _display
_raw = pd.read_csv(DATA_FILE)
_grade_results = []
def _filled(d):
    x=d.copy()
    cols=['wait_minutes','past_satisfaction_score','monthly_spend']
    x[cols]=x[cols].fillna(x[cols].median())
    x['contact_channel']=x.contact_channel.fillna(x.contact_channel.mode().iloc[0])
    return x
def _plot_ok(fig,h,k1,k2,title,ylabel,labels=None):
    a=fig.axes[0]
    ls=a.get_lines()
    labels=labels or [k1,k2]
    return (len(ls)==2 and a.get_title()==title and a.get_xlabel()=='Epoch' and a.get_ylabel()==ylabel
      and np.allclose(ls[0].get_ydata(),h.history[k1]) and np.allclose(ls[1].get_ydata(),h.history[k2])
      and a.get_legend() is not None and [t.get_text() for t in a.get_legend().get_texts()]==labels)
def _grade_q(number,checks):
    items=[]
    for label,expr in checks:
        try:
            ok=bool(eval(expr,globals()))
            items.append({'label':label,'passed':ok,'detail':'검사 통과' if ok else '실행 결과가 지시와 다릅니다. 변수명·조건·파라미터를 확인하세요.'})
        except Exception as e:
            items.append({'label':label,'passed':False,'detail':type(e).__name__+': '+str(e)[:180]})
    report={'schema':'aice-lab-v1','setId':SET_ID,'question':number,'checks':items}
    _grade_results.append(report)
    _display({'application/json':report},raw=True)
data=pd.read_csv(DATA_FILE)
clean = data.drop_duplicates().copy()
clean = clean.loc[clean['wait_minutes'].isna() | (clean['wait_minutes'] < 300)].reset_index(drop=True)
num = ['wait_minutes','past_satisfaction_score','monthly_spend']
clean[num] = clean[num].fillna(clean[num].median())
clean['contact_channel'] = clean['contact_channel'].fillna(clean['contact_channel'].mode().iloc[0])
preset = pd.get_dummies(clean.drop(columns=['ticket_id','created_at','resolution_hours']), columns=['issue_type','contact_channel','membership_tier'], dtype=int)
X = preset.drop(columns='escalated')
y = preset['escalated']
X_train, X_valid, y_train, y_valid = train_test_split(X,y,test_size=0.3,random_state=7,stratify=y)
ss = StandardScaler()
X_train_scaled = ss.fit_transform(X_train)
X_valid_scaled = ss.transform(X_valid)


## 1. 로지스틱 학습 (7점)

lr에 LogisticRegression(max_iter=1000,random_state=7)을 생성하고 스케일링된 훈련 데이터로 학습하세요.

In [107]:
# 여기에 답안코드를 작성하고 실행하세요.
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=1000,
    random_state=7
)

lr.fit(X_train_scaled, y_train);

In [108]:
_grade_q(1, [['모델 설정', 'lr.get_params()["max_iter"]==1000 and lr.get_params()["random_state"]==7'], ['학습 완료', 'hasattr(lr,"coef_") and lr.n_features_in_==X_train.shape[1]']])

## 2. 클래스 예측 (7점)

lr로 검증 데이터의 클래스를 예측해 y_pred에 저장하세요.

In [109]:
# 여기에 답안코드를 작성하고 실행하세요.
y_pred = lr.predict(X_valid_scaled)

In [110]:
_grade_q(2, [['예측 클래스', 'np.array_equal(y_pred,lr.predict(X_valid_scaled))']])

## 3. 확률 열 선택 (7점)

lr의 클래스 1 확률을 1차원 y_proba에 저장하세요.

In [111]:
# 여기에 답안코드를 작성하고 실행하세요.
y_proba = lr.predict_proba(X_valid_scaled)[:,1]
y_proba

array([0.15455302, 0.09358921, 0.29853807, 0.01441922, 0.16183699,
       0.019852  , 0.60694151, 0.17267792, 0.14948821, 0.45668403,
       0.08080048, 0.06454206, 0.72348183, 0.09222937, 0.06712775,
       0.26289332, 0.83823251, 0.03541141, 0.38573719, 0.02887141,
       0.06721565, 0.46513372, 0.02120417, 0.08250412, 0.14865764,
       0.66364327, 0.24984939, 0.12498613, 0.04028629, 0.03825833,
       0.59343442, 0.1211896 , 0.22492228, 0.54885939, 0.03090063,
       0.59700353, 0.05519722, 0.00706139, 0.05523861, 0.18878951,
       0.48185032, 0.04274425, 0.05060686, 0.1216887 , 0.37850256,
       0.57758399, 0.29641714, 0.15356629, 0.02185515, 0.1208943 ,
       0.08740167, 0.03308807, 0.10576981, 0.33233024, 0.16093557,
       0.0358762 , 0.06169028, 0.11517949, 0.10141923, 0.11660023,
       0.28267941, 0.11385067, 0.04990413, 0.24818501, 0.01912146,
       0.47490996, 0.16556423, 0.03043212, 0.04618637, 0.62976293,
       0.02062112, 0.08572237, 0.15026442, 0.24116621, 0.36372

In [112]:
_grade_q(3, [['예측 확률', 'np.allclose(y_proba,lr.predict_proba(X_valid_scaled)[:,1]) and np.asarray(y_proba).ndim==1']])

## 4. 혼동행렬 (7점)

실제 y_valid와 y_pred의 혼동행렬을 cm에 저장하세요.

In [113]:
# 여기에 답안코드를 작성하고 실행하세요.
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_valid, y_pred)
cm

array([[135,   6],
       [ 25,  13]])

In [114]:
_grade_q(4, [['행렬 방향', 'np.array_equal(cm,confusion_matrix(y_valid,y_pred))']])

## 5. TP·FN 구분 (7점)

cm에서 TP를 tp에, FN을 fn에 저장하세요.

In [115]:
# 여기에 답안코드를 작성하고 실행하세요.
tp = cm[1][1]
fn = cm[1][0]

In [116]:
_grade_q(5, [['TP', 'tp==confusion_matrix(y_valid,y_pred)[1,1]'], ['FN', 'fn==confusion_matrix(y_valid,y_pred)[1,0]']])

## 6. 재현율 계산 (7점)

클래스 1의 재현율을 recall_answer에 저장하세요.

In [117]:
# 여기에 답안코드를 작성하고 실행하세요.
from sklearn.metrics import recall_score

recall_answer = recall_score(y_valid, y_pred)
recall_answer

0.34210526315789475

In [118]:
_grade_q(6, [['재현율', 'np.isclose(recall_answer,recall_score(y_valid,y_pred))']])

## 7. ROC-AUC (7점)

확률을 사용해 auc_answer를 계산하세요.

In [119]:
# 여기에 답안코드를 작성하고 실행하세요.
from sklearn.metrics import roc_auc_score

auc_answer = roc_auc_score(y_valid, y_proba)
auc_answer

0.8195222097797685

In [120]:
_grade_q(7, [['확률 기반 AUC', 'np.isclose(auc_answer,roc_auc_score(y_valid,lr.predict_proba(X_valid_scaled)[:,1]))']])

## 8. balanced 학습 (7점)

balanced_lr에 class_weight="balanced", max_iter=1000,random_state=7인 로지스틱 모델을 학습하세요.

In [121]:
# 여기에 답안코드를 작성하고 실행하세요.
balanced_lr = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=7
)

balanced_lr.fit(X_train_scaled, y_train);

In [122]:
_grade_q(8, [['balanced 설정', 'balanced_lr.class_weight=="balanced" and hasattr(balanced_lr,"coef_")']])

## 9. 결정 트리 (7점)

dt에 최대깊이5, 최소분할3, 시드7인 DecisionTreeClassifier를 생성하고 스케일 전 훈련으로 학습하세요.

In [123]:
# 여기에 답안코드를 작성하고 실행하세요.
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=3,
    random_state=7
)

dt.fit(X_train, y_train);

In [124]:
_grade_q(9, [['트리 설정', 'dt.max_depth==5 and dt.min_samples_split==3 and dt.random_state==7 and hasattr(dt,"tree_")']])

## 10. 랜덤 포레스트 (7점)

rf에 트리50개, 깊이5, 시드7인 RandomForestClassifier를 생성하고 스케일 전 훈련으로 학습하세요.

In [125]:
# 여기에 답안코드를 작성하고 실행하세요.
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    random_state=7
)

rf.fit(X_train, y_train);

In [126]:
_grade_q(10, [['포레스트 설정', 'rf.n_estimators==50 and rf.max_depth==5 and rf.random_state==7 and len(rf.estimators_)==50']])

## 11. GridSearchCV (7점)

gs에 DecisionTreeClassifier(random_state=7)를 넣고 깊이 [3,5], 최소분할 [2,5]를 3겹, scoring="accuracy",n_jobs=-1로 탐색·학습하세요.

In [127]:
# 여기에 답안코드를 작성하고 실행하세요.
from sklearn.model_selection import GridSearchCV

params = {
    "max_depth":[3,5],
    "min_samples_split":[2,5]
}

gs = GridSearchCV(
    DecisionTreeClassifier(random_state=7),
    params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

gs.fit(X_train, y_train);

In [128]:
_grade_q(11, [['탐색 설정', 'gs.cv==3 and gs.scoring=="accuracy" and gs.n_jobs==-1 and gs.param_grid=={"max_depth":[3,5],"min_samples_split":[2,5]}'], ['학습된 최적 모델', 'hasattr(gs,"best_estimator_")']])

## 12. 최적 모델 예측 (7점)

gs의 best_estimator_로 검증 데이터를 예측하여 best_pred에 저장하세요.

In [129]:
# 여기에 답안코드를 작성하고 실행하세요.
best_pred = gs.best_estimator_.predict(X_valid)
best_pred

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1,
       0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0,
       0, 0, 0])

In [130]:
_grade_q(12, [['최적 모델 예측', 'np.array_equal(best_pred,gs.best_estimator_.predict(X_valid))']])

## 13. 변수 중요도 (7점)

rf 중요도를 feature,importance 두 컬럼의 fi로 만들고 내림차순 상위10개를 남기세요.

In [131]:
# 여기에 답안코드를 작성하고 실행하세요.
fi = pd.DataFrame({
    "feature":X_train.columns,
    "importance":rf.feature_importances_
}).sort_values("importance", ascending=False).head(10)

In [132]:
_grade_q(13, [['변수 중요도 매핑', 'fi.reset_index(drop=True).equals(pd.DataFrame({"feature":X_train.columns,"importance":rf.feature_importances_}).sort_values("importance",ascending=False).head(10).reset_index(drop=True))']])

## 14. 지표 선택 (9점)

최적 트리의 accuracy를 best_acc에, 클래스1 F1을 best_f1에 저장하세요.

In [133]:
# 여기에 답안코드를 작성하고 실행하세요.
best_acc = accuracy_score(y_valid, best_pred)
best_f1 = f1_score(y_valid, best_pred)

In [134]:
_grade_q(14, [['정확도', 'np.isclose(best_acc,accuracy_score(y_valid,best_pred))'], ['F1', 'np.isclose(best_f1,f1_score(y_valid,best_pred,zero_division=0))']])

## 제출
검증 결과를 포함한 상태로 저장하세요. VS Code에서 실행이 중단되면 오류를 고친 뒤 다시 전체 실행하세요. 설치가 필요하면 사이트의 환경 설정 안내를 확인하세요.